# CAMS dust deduction · reference-station workflow
**Option 1 first · Option 2 fallback · editable station links**

This self-contained notebook proposes the nearest eligible regional-background
reference within **50 km**, then lets you review and change every link on a map.
Dust is estimated from observed PM at the reference, not subtracted directly
from CAMS. CAMS supplies the initial screening of potentially affected days.

**Quick start:** run all cells with `DEMO_MODE = True` to explore synthetic data.
For real data, set it to `False`, choose the pollutant/year/countries, provide
`DataExtract.csv`, and enable downloads or point to existing EEA/CAMS files.
On Colab use **Runtime → Run all**. Keep the Python kernel running to edit links.

### Scientific choices
| Item | Implementation |
|---|---|
| Option 1 | Regional-background candidate (`rural` + `background`), median of non-dust observations inside ±15 **calendar** days |
| Option 2 | Own background series, or urban/suburban background reference for traffic/industrial/hotspot recipients; ±3 calendar days |
| Multiple dust days | Exclude all screened dust days. Default: a separate fixed window for each affected day |
| No suitable regional site | Explicit Option 2 fallback; missing daily reference data do **not** trigger a silent method switch |
| All dust days | Quantify below and above the selected PM limit; evaluate exceedances afterwards |
| Missing or insufficient data | Preserve `NaN` and a reason; never relabel missing CAMS as clean |

This implements the **user-supplied guideline excerpt, §§5.2.2–5.2.3,
Boxes 7–11**. The excerpt also describes an episode-edge window for consecutive
days, while its later paragraph and Box 11 keep a per-day window. Both modes
are provided; `per_day` is the documented default. Option 3 (chemical
speciation) is outside this notebook.

**Tool parameters, not prescribed guideline rules:** the 50 km radii, minimum
clean-day counts, altitude filters, CAMS threshold and concentration cap.
`rural + background` is a candidate classification, not proof of regional
representativeness. Automatic links remain provisional until reviewed.

In [ ]:
import importlib.util
import subprocess
import sys

_packages = {
    'numpy': 'numpy', 'pandas': 'pandas', 'xarray': 'xarray', 'scipy': 'scipy',
    'pyarrow': 'pyarrow', 'netCDF4': 'netCDF4', 'requests': 'requests',
    'ipywidgets': 'ipywidgets>=8', 'ipyleaflet': 'ipyleaflet>=0.19',
    'matplotlib': 'matplotlib', 'cdsapi': 'cdsapi>=0.7',
}
_missing = [package for module, package in _packages.items()
            if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *_missing])
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass

import calendar
import gc
import hashlib
import html
import json
import math
import os
import re
import time
import warnings
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import xarray as xr
from scipy.spatial import cKDTree
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display, clear_output

ALGORITHM_VERSION = 'reference-dust-1.0'

## 1 · Settings
`DOWNLOAD_* = False` reuses existing files. `REFRESH_DOWNLOADS = True` replaces
matching downloaded archives. CAMS credentials are requested privately only
when downloads are enabled; no API key is stored in the notebook or exports.
Accept the CAMS dataset licence in your ADS account first.

EEA daily products retain their reported fixed timezone. Hourly EEA stamps are
interpreted as fixed UTC+1, converted to UTC, then aggregated on each station's
reported standard day. **Both paths therefore use local-standard days here**,
unlike the UTC-day hourly path of the original notebook. CAMS is hourly UTC.
Cross-timezone daily transfers are excluded because the daily windows differ.

In [ ]:
DEMO_MODE = True                 # Switch to False for your real files.
YEAR = 2024
POLLUTANT = 'PM10'               # 'PM10' or 'PM2.5'
DATASET = 'E1a'                  # 'E1a' verified; 'E2a' up-to-date
EEA_TEMPORAL_FLAG = 'day'        # 'day' or 'hour'
COUNTRIES = ['ES']

PROJECT_DIR = Path('./CAMS_reference_tool')
USE_GOOGLE_DRIVE = False
GDRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/CAMS_Tool_output')
METADATA_PATH = None             # e.g. Path('/content/DataExtract.csv')
EEA_INPUT_DIR = None             # existing Parquet folder, recursively searched
CAMS_INPUT_DIR = None            # existing hourly CAMS NetCDF folder
DOWNLOAD_EEA = False
DOWNLOAD_CAMS = False
REFRESH_DOWNLOADS = False
REBUILD_CAMS_CACHE = False

MIN_STATION_COVERAGE_PCT = 65.0
MIN_OBS_HOURS = 18               # distinct valid hours per observed local day
MIN_CAMS_HOURS = 18              # distinct valid hourly CAMS samples per local day
CAMS_DUST_THRESHOLD = 5.0        # µg/m³; requires model-specific validation
CAMS_EXPECTED_STEP_HOURS = 1     # this notebook expects hourly regional reanalysis
CAMS_BUFFER_DEG = 0.2
CAMS_UNITS_OVERRIDE = None       # only if the original NetCDF has no units
PM_DAILY_LIMIT = {'PM10': 50.0, 'PM2.5': 25.0}[POLLUTANT]  # analysis setting

OPTION1_RADIUS_KM = 50.0
OPTION2_RADIUS_KM = 50.0
MAX_ALTITUDE_DIFFERENCE_M = None  # e.g. 300; None means no automatic filter
SAME_COUNTRY_ONLY = False
MIN_CLEAN_DAYS_OPTION1 = 10       # engineering default, not a guideline minimum
MIN_CLEAN_DAYS_OPTION2 = 3        # engineering default, not a guideline minimum
MIN_CLEAN_DAYS_EACH_SIDE = 1      # configurable, not stipulated in the excerpt
OPTION2_WINDOW_MODE = 'per_day'  # 'per_day' or 'episode_edges'
REQUIRE_TARGET_DUST_SCREEN = True
CAP_DEDUCTION_TO_OBSERVED_PM = True  # retain original estimate + cap flag

# Optional daily evidence decisions. Template exported later.
# Columns: Samplingpoint, day, is_dust_candidate (0/1), evidence_note
EPISODE_OVERRIDES_CSV = None
# Optional edited station eligibility table exported by the map.
STATION_RULES_CSV = None
AUTO_LOAD_SAVED_LINKS = True

assert POLLUTANT in {'PM10', 'PM2.5'}
assert DATASET in {'E1a', 'E2a'} and EEA_TEMPORAL_FLAG in {'hour', 'day'}
assert OPTION2_WINDOW_MODE in {'per_day', 'episode_edges'}
assert 0 < MIN_OBS_HOURS <= 24 and 0 < MIN_CAMS_HOURS <= 24
assert MIN_CLEAN_DAYS_OPTION1 <= 30 and MIN_CLEAN_DAYS_OPTION2 <= 6

CONTEXT_START = pd.Timestamp(YEAR, 1, 1) - pd.Timedelta(days=15)
CONTEXT_END = pd.Timestamp(YEAR, 12, 31) + pd.Timedelta(days=15)
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
if USE_GOOGLE_DRIVE and not DEMO_MODE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
OUTPUT_ROOT = GDRIVE_OUTPUT_DIR if USE_GOOGLE_DRIVE and not DEMO_MODE else PROJECT_DIR
RUN_DIR = OUTPUT_ROOT / ('demo' if DEMO_MODE else f'{DATASET}_{POLLUTANT}_{YEAR}_{EEA_TEMPORAL_FLAG}')
RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = OUTPUT_ROOT / 'IRA_dust' / 'reference_cache_v1'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"{'SYNTHETIC DEMO' if DEMO_MODE else 'REAL DATA'} · {YEAR} · {POLLUTANT}")
print('Results:', RUN_DIR.resolve())

## 2 · Download or reuse input data
The observation request includes 15 context days on both sides of the year,
plus one UTC boundary day. CAMS requests include December of the preceding
year and January of the following year. Missing context remains missing if
those data have not yet been published; it is never replaced with another year.

Existing zip names from the original notebook are recognised. Files are
extracted locally. No large full-year CAMS grid is loaded in memory.

In [ ]:
EEA_API_URL = 'https://eeadmz1-downloads-api-appservice.azurewebsites.net/'
CAMS_DATASET = 'cams-europe-air-quality-reanalyses'

def atomic_json(path, data):
    def compatible(value):
        if isinstance(value, dict):
            return {str(k): compatible(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return [compatible(v) for v in value]
        if isinstance(value, (np.integer, np.floating, np.bool_)):
            value = value.item()
        if isinstance(value, float) and not math.isfinite(value):
            return None
        if value is pd.NA or value is pd.NaT:
            return None
        if isinstance(value, (pd.Timestamp, datetime)):
            return value.isoformat()
        return value
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(compatible(data), indent=2, ensure_ascii=False, default=str, allow_nan=False), encoding='utf-8')
    tmp.replace(path)

def safe_extract(archive, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f'Unsafe archive member: {member.filename}')
        z.extractall(destination)

def download_eea_inputs():
    base_folder = PROJECT_DIR / f'EEA_{POLLUTANT}' / str(YEAR) / DATASET / EEA_TEMPORAL_FLAG
    body = {
        'countries': COUNTRIES, 'cities': [], 'pollutants': [POLLUTANT],
        'dataset': {'E2a': 1, 'E1a': 2}[DATASET],
        'dateTimeStart': (CONTEXT_START - pd.Timedelta(days=1)).strftime('%Y-%m-%dT00:00:00Z'),
        'dateTimeEnd': (CONTEXT_END + pd.Timedelta(days=1)).strftime('%Y-%m-%dT23:59:59Z'),
        'aggregationType': EEA_TEMPORAL_FLAG, 'email': '',
    }
    request_hash = hashlib.sha256(json.dumps(body, sort_keys=True).encode()).hexdigest()[:12]
    folder = Path(EEA_INPUT_DIR) if EEA_INPUT_DIR else base_folder / f'selection_{request_hash}'
    folder.mkdir(parents=True, exist_ok=True)
    archive_dir = OUTPUT_ROOT / f'EEA_{POLLUTANT}' / EEA_TEMPORAL_FLAG
    archive_dir.mkdir(parents=True, exist_ok=True)
    archive = archive_dir / f'{DATASET}_{POLLUTANT}_{EEA_TEMPORAL_FLAG}_{YEAR}_{request_hash}.zip'
    if DOWNLOAD_EEA and (REFRESH_DOWNLOADS or not archive.exists()):
        response = requests.post(EEA_API_URL + 'ParquetFile/async', json=body, timeout=120)
        response.raise_for_status()
        try:
            payload = response.json()
        except ValueError:
            payload = response.text.strip().strip('"')
        if isinstance(payload, str):
            url = payload
        elif isinstance(payload, dict):
            url = payload.get('downloadUrl') or payload.get('url')
        else:
            url = None
        if not url or not url.startswith('https://'):
            raise RuntimeError('EEA did not return a valid HTTPS download URL; inspect the API response privately.')
        started = time.monotonic()
        while True:
            with requests.get(url, stream=True, timeout=120) as data:
                if data.status_code not in (202, 404):
                    data.raise_for_status()
                    tmp = archive.with_suffix('.zip.part')
                    with tmp.open('wb') as f:
                        for chunk in data.iter_content(2**20):
                            f.write(chunk)
                    if not zipfile.is_zipfile(tmp):
                        tmp.unlink(missing_ok=True)
                        raise RuntimeError('EEA response was not a ZIP archive.')
                    tmp.replace(archive)
                    atomic_json(archive.with_suffix('.request.json'), body)
                    break
            if time.monotonic() - started > 3600:
                raise TimeoutError('EEA archive not ready within one hour.')
            print('EEA is preparing the archive …', flush=True)
            time.sleep(15)
    if archive.exists():
        safe_extract(archive, folder)
    elif not list(folder.rglob('*.parquet')):
        legacy = archive_dir / f'{DATASET}_{POLLUTANT}_{EEA_TEMPORAL_FLAG}_{YEAR}.zip'
        if legacy.exists():
            warnings.warn('Using an older EEA archive; it may lack January context after the target year.')
            safe_extract(legacy, folder)
        else:
            raise FileNotFoundError('No EEA data. Enable DOWNLOAD_EEA or set EEA_INPUT_DIR.')
    return folder

def download_cams_inputs():
    raw = Path(CAMS_INPUT_DIR) if CAMS_INPUT_DIR else PROJECT_DIR / 'IRA_dust'
    raw.mkdir(parents=True, exist_ok=True)
    store = OUTPUT_ROOT / 'IRA_dust'
    store.mkdir(parents=True, exist_ok=True)
    months = pd.period_range(CONTEXT_START, CONTEXT_END, freq='M')
    # Reuse archives made by the original notebook, including quarterly files.
    legacy = set(store.glob(f'CAMS_IRA_{YEAR}*.zip')) | set(raw.glob(f'CAMS_IRA_{YEAR}*.zip'))
    legacy |= set(store.glob(f'CAMS_IRA_{YEAR-1}_12.zip')) | set(raw.glob(f'CAMS_IRA_{YEAR-1}_12.zip'))
    for archive in sorted(legacy):
        safe_extract(archive, raw)
    if DOWNLOAD_CAMS:
        import cdsapi
        # Use configured ~/.cdsapirc or private prompt; never print or export keys.
        try:
            client = cdsapi.Client(quiet=True)
        except Exception:
            from getpass import getpass
            key = getpass('ADS personal access token (hidden): ')
            client = cdsapi.Client(url='https://ads.atmosphere.copernicus.eu/api', key=key, quiet=True)
            del key
        for month in months:
            archive = store / f'CAMS_IRA_{month.year}_{month.month:02d}.zip'
            if REFRESH_DOWNLOADS or not archive.exists():
                print('Requesting hourly CAMS:', str(month), flush=True)
                tmp = archive.with_suffix('.zip.part')
                client.retrieve(CAMS_DATASET, {
                    'variable': ['dust'], 'model': ['ensemble'], 'level': ['0'],
                    'type': ['interim_reanalysis'], 'year': [str(month.year)],
                    'month': [f'{month.month:02d}'],
                }).download(str(tmp))
                if not zipfile.is_zipfile(tmp):
                    raise RuntimeError('CAMS did not return the expected ZIP. Check the ADS request form.')
                tmp.replace(archive)
            safe_extract(archive, raw)
    else:
        for month in months:
            archive = store / f'CAMS_IRA_{month.year}_{month.month:02d}.zip'
            if archive.exists() and archive not in legacy:
                safe_extract(archive, raw)
    paths = sorted(p for p in raw.rglob('*.nc') if 'dust' in p.name.lower())
    if not paths:
        print('No hourly CAMS NetCDF on disk. A matching daily cache will be tried.')
    return paths

## 3 · Metadata and observed daily means
Put `DataExtract.csv` in the output/project folder or set `METADATA_PATH`.
Required fields are sampling-point ID, coordinates, station area/type and
timezone. Unknown timezone labels are **not** silently set to UTC.
Conflicting station metadata or duplicate observations stop the run for review.
Historical station moves/timezone changes must be resolved in the metadata.

In [ ]:
def parse_offset_minutes(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().upper().replace(' ', '')
    text = text.rsplit('/', 1)[-1]
    if text in {'UTC', 'Z', 'UTCZ'}:
        return 0
    match = re.fullmatch(r'UTC([+-])(\d{1,2})(?::?(\d{2}))?', text)
    if not match:
        return np.nan
    hours, minutes = int(match[2]), int(match[3] or 0)
    if hours > 14 or minutes >= 60 or (hours == 14 and minutes):
        return np.nan
    return (1 if match[1] == '+' else -1) * (60 * hours + minutes)

def category_label(value):
    return str(value).strip().lower().rsplit('/', 1)[-1].replace('_', ' ')

def find_column(frame, options, required=True):
    lookup = {str(c).strip().lower(): c for c in frame.columns}
    for name in options:
        if name.lower() in lookup:
            return lookup[name.lower()]
    if required:
        raise KeyError(f'Missing metadata field: expected one of {options}')
    return None

def unit_factor_to_ug(value):
    text = str(value).strip().lower().replace('μ', 'u').replace('µ', 'u')
    text = re.sub(r'[\s^*/{}()·.]', '', text).replace('³', '3').replace('−', '-').replace('⁻', '-')
    if text in {'ugm-3', 'ugm3', 'ugper m3', 'ugpermetrecubed', 'ugperm3'}:
        return 1.0
    if text in {'mgm-3', 'mgm3', 'mgperm3'}:
        return 1e3
    if text in {'kgm-3', 'kgm3', 'kgperm3'}:
        return 1e9
    if text in {'gm-3', 'gm3', 'gperm3'}:
        return 1e6
    raise ValueError(f'Unknown concentration unit {value!r}; supply an explicit supported unit.')

def load_metadata(path):
    raw = pd.read_csv(path, sep=None, engine='python', dtype=str)
    pollutant_col = find_column(raw, ['Air Pollutant', 'AirPollutant', 'Pollutant'], required=False)
    if pollutant_col:
        normalized = raw[pollutant_col].astype(str).str.upper().str.replace(' ', '', regex=False)
        accepted = {POLLUTANT.upper(), '5' if POLLUTANT == 'PM10' else '6001'}
        subset = raw.loc[normalized.str.rsplit('/', n=1).str[-1].isin(accepted)]
        if subset.empty:
            raise ValueError(f'No metadata for {POLLUTANT}; do not mix pollutants when resolving sampling points.')
        raw = subset.copy()
    out = pd.DataFrame(index=raw.index)
    id_col = find_column(raw, ['Samplingpoint', 'Samplingpoint_id'], required=False)
    if id_col:
        out['Samplingpoint'] = raw[id_col].str.strip()
    else:
        station_col = find_column(raw, ['Air Quality Station EoI Code'])
        point_col = find_column(raw, ['Sampling Point Id'])
        country = raw[station_col].str[:2]
        point = raw[point_col].str.strip()
        out['Samplingpoint'] = np.where(point.str.match(r'^[A-Z]{2}/', na=False), point, country + '/' + point)
    for dst, names in {
        'Latitude': ['Latitude'], 'Longitude': ['Longitude'], 'Altitude': ['Altitude'],
        'Area': ['Air Quality Station Area', 'Area'],
        'Type': ['Air Quality Station Type', 'Type'],
        'Timezone': ['Timezone', 'Time Zone'],
    }.items():
        out[dst] = raw[find_column(raw, names)]
    for c in ['Latitude', 'Longitude', 'Altitude']:
        out[c] = pd.to_numeric(out[c], errors='coerce')
    out['Area'] = out['Area'].map(category_label)
    out['Type'] = out['Type'].map(category_label)
    out['tz_offset_min'] = out['Timezone'].map(parse_offset_minutes)
    out['Country'] = out['Samplingpoint'].str[:2]
    fields = ['Latitude', 'Longitude', 'Area', 'Type', 'tz_offset_min']
    conflicts = out.groupby('Samplingpoint')[fields].nunique(dropna=False).gt(1).any(axis=1)
    if conflicts.any():
        raise ValueError('Conflicting metadata for: ' + ', '.join(conflicts[conflicts].index[:10]))
    out = out.dropna(subset=['Samplingpoint']).drop_duplicates('Samplingpoint').reset_index(drop=True)
    out['is_hotspot'] = False  # may be revised in the rules CSV/map
    out['regional_enabled'] = out['Area'].eq('rural') & out['Type'].eq('background')
    out['option2_enabled'] = out['Type'].eq('background') & out['Area'].isin(['rural', 'suburban', 'urban'])
    return out

def to_utc_from_eea(stamps):
    # Fixed UTC+1 for naive hourly EEA export; honour explicit offsets if present.
    def one(x):
        if pd.isna(x):
            return pd.NaT
        t = pd.Timestamp(x)
        return t.tz_localize('Etc/GMT-1').tz_convert('UTC') if t.tzinfo is None else t.tz_convert('UTC')
    return pd.to_datetime(stamps.map(one), utc=True, errors='coerce')

def local_calendar_dates(stamps):
    # Drop an explicitly encoded timezone without shifting local wall time.
    def one(x):
        if pd.isna(x):
            return pd.NaT
        try:
            t = pd.Timestamp(x)
            if t.tzinfo is not None:
                t = t.tz_localize(None)
            return t.normalize()
        except (ValueError, TypeError):
            return pd.NaT
    return pd.to_datetime(stamps.map(one), errors='coerce')

def observed_daily_from_frame(raw, metadata, temporal_flag):
    required = ['Value', 'Validity', 'Verification', 'AggType', 'Start', 'Samplingpoint', 'Unit']
    absent = set(required) - set(raw)
    if absent:
        raise KeyError(f'EEA data missing: {sorted(absent)}')
    data = raw.copy()
    if 'Pollutant' in data:
        p = data['Pollutant'].astype(str).str.upper().str.replace(' ', '', regex=False)
        # Numeric EEA pollutant IDs 5 and 6001 are also accepted.
        accepted = {POLLUTANT.upper(), '5' if POLLUTANT == 'PM10' else '6001'}
        if not p.isin(accepted).all():
            raise ValueError('Unexpected pollutant in EEA input; use a pollutant-specific folder.')
    data['Value'] = pd.to_numeric(data['Value'], errors='coerce')
    good = (pd.to_numeric(data['Validity'], errors='coerce').eq(1)
            & pd.to_numeric(data['Verification'], errors='coerce').lt(3)
            & data['AggType'].astype(str).str.lower().eq(temporal_flag)
            & data['Value'].ge(0) & np.isfinite(data['Value']))
    data = data.loc[good].copy()
    factors = data['Unit'].map(lambda x: unit_factor_to_ug(x))
    data['Value'] = (data['Value'] * factors).astype('float32')
    data['Samplingpoint'] = data['Samplingpoint'].astype(str)
    data = data.drop_duplicates(['Samplingpoint', 'Start', 'Value'])
    if data.duplicated(['Samplingpoint', 'Start']).any():
        raise ValueError('Conflicting EEA values for the same sampling point and timestamp.')
    if temporal_flag == 'day':
        data['day'] = local_calendar_dates(data['Start'])
        if data.duplicated(['Samplingpoint', 'day']).any():
            raise ValueError('Several daily EEA observations for the same station/date.')
        daily = data.rename(columns={'Value': 'daily_mean'})[['Samplingpoint', 'day', 'daily_mean']].copy()
        daily['n_obs_hours'] = np.nan  # daily product does not expose hourly coverage
    else:
        data['utc'] = to_utc_from_eea(data['Start'])
        offsets = metadata.set_index('Samplingpoint')['tz_offset_min']
        data['offset'] = data['Samplingpoint'].map(offsets)
        data = data.dropna(subset=['utc', 'offset'])
        shifted = data['utc'].dt.tz_localize(None) + pd.to_timedelta(data['offset'], unit='min')
        data['day'] = shifted.dt.floor('D')
        if data.duplicated(['Samplingpoint', 'utc']).any():
            raise ValueError('Duplicate UTC hours in EEA data.')
        daily = data.groupby(['Samplingpoint', 'day'], observed=True).agg(
            daily_mean=('Value', 'mean'), n_obs_hours=('utc', 'nunique')).reset_index()
        daily = daily.loc[daily['n_obs_hours'].ge(MIN_OBS_HOURS)].copy()
    daily = daily.loc[daily['day'].between(CONTEXT_START, CONTEXT_END)].dropna(subset=['day'])
    daily['unit'] = 'µg/m³'
    return daily.reset_index(drop=True)

def load_eea_observations(folder, metadata):
    frames = []
    seen_stations = set()
    for path in sorted(Path(folder).rglob('*.parquet')):
        names = pq.ParquetFile(path).schema.names
        needed = ['Value', 'Validity', 'Verification', 'AggType', 'Start', 'Samplingpoint', 'Unit', 'Pollutant']
        frame = pd.read_parquet(path, columns=[c for c in needed if c in names])
        ids = set(frame['Samplingpoint'].dropna().astype(str))
        if seen_stations.intersection(ids):
            raise ValueError('Several EEA input files contain the same sampling point. Consolidate its raw time series or remove old extractions before running; hourly boundaries must be aggregated together.')
        seen_stations.update(ids)
        # EEA exports one complete time series per sampling point/file.
        # Reduce each file before concatenating: do not load all European hours.
        frames.append(observed_daily_from_frame(frame, metadata, EEA_TEMPORAL_FLAG))
        del frame
    if not frames:
        raise FileNotFoundError(f'No EEA Parquet under {folder}')
    daily = pd.concat(frames, ignore_index=True)
    del frames
    gc.collect()
    return daily

def restrict_station_coverage(daily, stations):
    mask = daily['day'].dt.year.eq(YEAR) & daily['daily_mean'].notna()
    coverage = daily.loc[mask].groupby('Samplingpoint')['day'].nunique() / (366 if calendar.isleap(YEAR) else 365) * 100
    good_ids = coverage[coverage.ge(MIN_STATION_COVERAGE_PCT)].index
    stations = stations.loc[stations['Samplingpoint'].isin(good_ids)].copy()
    stations['coverage_pct'] = stations['Samplingpoint'].map(coverage)
    daily = daily.loc[daily['Samplingpoint'].isin(stations['Samplingpoint'])].copy()
    if stations.empty:
        raise ValueError('No stations meet target-year coverage. Check inputs/year and the coverage setting.')
    return daily, stations.reset_index(drop=True)

## 4 · Hourly CAMS → local daily values at stations
Bilinear interpolation is performed on a one-day grid subset. Monthly/file
station-hourly caches are streamed as Parquet row groups. Local daily sums
retain boundary hours across months and years. No full-year grid or full-year
station-hourly table is assembled. CAMS units are checked before conversion.

Cache keys include source path/size/mtime, station coordinates and algorithm
version. Daily keys include timezones, date range, coverage and interpolation
settings. Use `REBUILD_CAMS_CACHE` after modifying files without changing their
size/mtime. With no raw grids, only an exactly matching daily cache is reused.

In [ ]:
def digest(payload):
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:20]

def coord_name(da, prefix):
    names = [d for d in da.dims if str(d).lower().startswith(prefix)]
    if len(names) != 1:
        raise ValueError(f'Expected one {prefix} dimension, got {da.dims}')
    return names[0]

def dust_array(ds):
    names = ['dust'] if 'dust' in ds else [v for v in ds.data_vars if 'dust' in v.lower()]
    if len(names) != 1:
        raise ValueError(f'Cannot resolve dust variable in {list(ds.data_vars)}')
    da = ds[names[0]]
    t, y, x = (coord_name(da, p) for p in ['time', 'lat', 'lon'])
    for dim in list(da.dims):
        if dim not in {t, y, x}:
            if da.sizes[dim] != 1:
                raise ValueError(f'Expected surface-only data, found {dim}={da.sizes[dim]}')
            da = da.isel({dim: 0}, drop=True)
    if da[y].ndim != 1 or da[x].ndim != 1:
        raise ValueError('Only rectilinear CAMS grids are supported.')
    return da, t, y, x

def interpolation_window(values, low, high):
    values = np.asarray(values)
    indices = np.flatnonzero((values >= low - CAMS_BUFFER_DEG) & (values <= high + CAMS_BUFFER_DEG))
    if not len(indices):
        indices = np.array([np.argmin(abs(values - (low + high) / 2))])
    return slice(max(0, indices.min() - 1), min(len(values), indices.max() + 2))

def source_fingerprints(paths):
    return [{'path': str(Path(p).resolve()), 'bytes': Path(p).stat().st_size,
             'mtime_ns': Path(p).stat().st_mtime_ns} for p in paths]

def grid_station_blocks(path, stations):
    with xr.open_dataset(path) as ds:
        da, t, y, x = dust_array(ds)
        units = CAMS_UNITS_OVERRIDE or da.attrs.get('units') or da.attrs.get('unit')
        factor = unit_factor_to_ug(units)
        lons = np.asarray(da[x].values, float)
        lats = np.asarray(da[y].values, float)
        station_lon = (stations['Longitude'].to_numpy(float) - lons.min()) % 360 + lons.min()
        station_lat = stations['Latitude'].to_numpy(float)
        inside = (np.isfinite(station_lon) & np.isfinite(station_lat)
                  & (station_lon <= lons.max()) & (station_lat >= lats.min()) & (station_lat <= lats.max()))
        if not inside.any():
            return
        da = da.isel({y: interpolation_window(lats, station_lat[inside].min(), station_lat[inside].max()),
                      x: interpolation_window(lons, station_lon[inside].min(), station_lon[inside].max())})
        times = pd.DatetimeIndex(pd.to_datetime(da[t].values, utc=True)).tz_localize(None)
        if times.has_duplicates:
            raise ValueError(f'Duplicate CAMS timestamps inside {path}')
        context = (times >= CONTEXT_START - pd.Timedelta(days=1)) & (times < CONTEXT_END + pd.Timedelta(days=2))
        indices = np.flatnonzero(context)
        if not len(indices):
            return
        if not times.equals(times.floor('h')):
            raise ValueError('CAMS timestamps are not on an hourly grid.')
        if len(times) > 1:
            delta = np.diff(times.sort_values().asi8) / 3.6e12
            if np.median(delta) != 1:
                raise ValueError('Expected hourly CAMS input. Daily grids / sparse snapshots cannot be used for local-day means.')
        lat_targets = xr.DataArray(station_lat[inside], dims='station')
        lon_targets = xr.DataArray(station_lon[inside], dims='station')
        for day in times[indices].normalize().unique().sort_values():
            positions = np.flatnonzero(context & (times.normalize() == day))
            sub = da.isel({t: positions}).astype('float32').load()
            result = sub.interp({y: lat_targets, x: lon_targets}, method='linear').transpose(t, 'station')
            values = np.full((len(positions), len(stations)), np.nan, dtype='float32')
            values[:, inside] = np.asarray(result.values, dtype='float32') * factor
            values[~np.isfinite(values) | (values < 0)] = np.nan
            frame = pd.DataFrame(values, columns=stations['Samplingpoint'].astype(str))
            frame.insert(0, 'time', times[positions])
            yield frame
            del sub, result, values, frame

def cached_station_blocks(path, stations):
    payload = {
        'version': ALGORITHM_VERSION, 'source': source_fingerprints([path]),
        'stations': stations[['Samplingpoint', 'Latitude', 'Longitude']].to_dict('records'),
        'buffer': CAMS_BUFFER_DEG, 'units_override': CAMS_UNITS_OVERRIDE,
        'start': CONTEXT_START, 'end': CONTEXT_END,
    }
    cache = CACHE_DIR / f'hourly_{digest(payload)}.parquet'
    if REBUILD_CAMS_CACHE or not cache.exists():
        tmp = cache.with_suffix('.parquet.part')
        writer = None
        try:
            for block in grid_station_blocks(path, stations):
                table = pa.Table.from_pandas(block, preserve_index=False)
                if writer is None:
                    writer = pq.ParquetWriter(tmp, table.schema, compression='snappy')
                writer.write_table(table)
            if writer is not None:
                writer.close()
                writer = None
                tmp.replace(cache)
        finally:
            if writer is not None:
                writer.close()
            tmp.unlink(missing_ok=True)
    if cache.exists():
        for batch in pq.ParquetFile(cache).iter_batches(batch_size=24):
            yield batch.to_pandas()

def cams_daily_at_stations(paths, stations):
    columns = ['Samplingpoint', 'Latitude', 'Longitude', 'tz_offset_min']
    ordered = stations.sort_values('Samplingpoint').reset_index(drop=True)
    payload = {
        'version': ALGORITHM_VERSION, 'stations': ordered[columns].to_dict('records'),
        'start': CONTEXT_START, 'end': CONTEXT_END, 'min_hours': MIN_CAMS_HOURS,
        'step': CAMS_EXPECTED_STEP_HOURS, 'buffer': CAMS_BUFFER_DEG,
        'units_override': CAMS_UNITS_OVERRIDE,
    }
    path = CACHE_DIR / f'local_daily_{digest(payload)}.parquet'
    meta_path = path.with_suffix('.json')
    sources = source_fingerprints(paths)
    if path.exists() and meta_path.exists() and not REBUILD_CAMS_CACHE:
        old = json.loads(meta_path.read_text())
        if not paths or old.get('sources') == sources:
            print('Reusing matching local-daily CAMS cache' + (' without raw grids.' if not paths else '.'))
            return pd.read_parquet(path)
    if not paths:
        raise FileNotFoundError('No matching CAMS daily cache and no raw NetCDF. Enable DOWNLOAD_CAMS or set CAMS_INPUT_DIR.')
    if CAMS_EXPECTED_STEP_HOURS != 1:
        raise ValueError('Only hourly CAMS is supported; do not interpret sparse snapshots as hourly coverage.')
    days = pd.date_range(CONTEXT_START, CONTEXT_END, freq='D')
    shape = (len(days), len(ordered))
    sums = np.zeros(shape, dtype='float64')
    counts = np.zeros(shape, dtype='int16')
    offsets = ordered['tz_offset_min'].to_numpy(float)
    seen = set()
    duplicate_hours = 0
    for k, source in enumerate(paths, 1):
        print(f'CAMS file {k}/{len(paths)}: {Path(source).name}', flush=True)
        for block in cached_station_blocks(source, ordered):
            stamps = pd.DatetimeIndex(block['time'])
            keep = np.array([t.value not in seen for t in stamps])
            duplicate_hours += int((~keep).sum())
            for t in stamps[keep]:
                seen.add(t.value)
            if not keep.any():
                continue
            stamps = stamps[keep]
            values = block.loc[keep, ordered['Samplingpoint'].tolist()].to_numpy(dtype='float32')
            for offset in np.unique(offsets[np.isfinite(offsets)]):
                cols = np.flatnonzero(offsets == offset)
                local = (stamps + pd.to_timedelta(offset, unit='min')).normalize()
                rows = (local - CONTEXT_START).days.to_numpy()
                good = (rows >= 0) & (rows < len(days))
                vals = values[good][:, cols]
                valid = np.isfinite(vals)
                rr = rows[good, None]
                cc = cols[None, :]
                np.add.at(sums, (rr, cc), np.where(valid, vals, 0))
                np.add.at(counts, (rr, cc), valid.astype('int16'))
        gc.collect()
    if duplicate_hours:
        warnings.warn(f'{duplicate_hours} overlapping source hours ignored; first source in sorted path order wins. Avoid mixing reanalysis versions.')
    average = np.full(shape, np.nan, dtype='float32')
    np.divide(sums, counts, out=average, where=counts >= MIN_CAMS_HOURS)
    out = pd.DataFrame({
        'day': np.repeat(days.to_numpy(), len(ordered)),
        'Samplingpoint': np.tile(ordered['Samplingpoint'].to_numpy(), len(days)),
        'cams_dust': average.reshape(-1), 'n_cams_hours': counts.reshape(-1),
    })
    out.to_parquet(path, index=False)
    atomic_json(meta_path, {**payload, 'sources': sources, 'duplicate_hours_ignored': duplicate_hours})
    return out

## 5 · Build the station/day table
The synthetic example includes Option 1 links, Option 2 fallback sites, a site
without a suitable reference, missing data and a long episode that can exhaust
a short clean-day window. **Demo values are illustrative, not EEA observations.**

In [ ]:
def synthetic_inputs():
    rows = [
        ('NL/RB_UTRECHT', 52.10, 5.18, 5, 'rural', 'background'),
        ('NL/RB_EAST', 52.18, 5.70, 12, 'rural', 'background'),
        ('NL/UB_UTRECHT', 52.09, 5.12, 5, 'urban', 'background'),
        ('NL/TRAFFIC_A', 52.08, 5.11, 5, 'urban', 'traffic'),
        ('NL/INDUSTRY_A', 52.03, 5.20, 5, 'suburban', 'industrial'),
        ('NL/TRAFFIC_EAST', 52.18, 5.50, 8, 'urban', 'traffic'),
        ('NL/UB_NORTH', 53.22, 6.57, 5, 'urban', 'background'),
        ('NL/TRAFFIC_NORTH', 53.23, 6.59, 5, 'urban', 'traffic'),
        ('NL/SUB_NORTH', 53.15, 6.60, 6, 'suburban', 'background'),
        ('NL/ISOLATED', 52.70, 7.75, 5, 'rural', 'traffic'),
    ]
    stations = pd.DataFrame(rows, columns=['Samplingpoint', 'Latitude', 'Longitude', 'Altitude', 'Area', 'Type'])
    stations['Country'] = 'NL'
    stations['Timezone'] = 'UTC+01'
    stations['tz_offset_min'] = 60
    stations['is_hotspot'] = False
    stations['regional_enabled'] = stations['Area'].eq('rural') & stations['Type'].eq('background')
    stations['option2_enabled'] = stations['Type'].eq('background')
    days = pd.date_range(CONTEXT_START, CONTEXT_END)
    rng = np.random.default_rng(271)
    pieces = []
    for i, s in stations.iterrows():
        day_of_year = days.dayofyear.to_numpy()
        baseline = 13 + 3 * np.cos(day_of_year / 366 * 2 * np.pi)
        baseline += {'background': 0, 'traffic': 12, 'industrial': 8}[s.Type]
        if s.Area == 'urban':
            baseline += 4
        dust = np.zeros(len(days))
        for a, b, load in [(f'{YEAR}-03-18', f'{YEAR}-03-20', 24),
                           (f'{YEAR}-06-08', f'{YEAR}-06-08', 7),
                           (f'{YEAR}-08-10', f'{YEAR}-08-18', 29),
                           (f'{YEAR}-11-04', f'{YEAR}-11-05', 18)]:
            dust[(days >= a) & (days <= b)] = load
        dust *= 1 + 0.03 * i
        measured = np.maximum(0, baseline + dust + rng.normal(0, 1.2, len(days)))
        modeled = np.maximum(0, dust * 0.65 + rng.normal(0, 0.3, len(days)))
        part = pd.DataFrame({'Samplingpoint': s.Samplingpoint, 'day': days,
                             'daily_mean': measured.astype('float32'),
                             'cams_dust': modeled.astype('float32'), 'n_cams_hours': 24,
                             'n_obs_hours': 24, 'unit': 'µg/m³'})
        if i == 0:
            part.loc[part.day.eq(pd.Timestamp(YEAR, 3, 19)), 'daily_mean'] = np.nan
        if i == 2:
            part.loc[part.day.eq(pd.Timestamp(YEAR, 6, 8)), 'cams_dust'] = np.nan
        pieces.append(part)
    return pd.concat(pieces, ignore_index=True), stations

def apply_station_rules(stations, path):
    if path is None:
        return stations
    rules = pd.read_csv(path, dtype={'Samplingpoint': str})
    if rules['Samplingpoint'].duplicated().any():
        raise ValueError('Duplicate IDs in station rules.')
    out = stations.set_index('Samplingpoint').copy()
    for row in rules.to_dict('records'):
        sid = row['Samplingpoint']
        if sid not in out.index:
            continue
        for col in ['regional_enabled', 'option2_enabled', 'is_hotspot']:
            if col in row and pd.notna(row[col]):
                text = str(row[col]).strip().lower()
                if text not in {'1', '0', 'true', 'false'}:
                    raise ValueError(f'Invalid boolean in rules: {col}={row[col]}')
                out.loc[sid, col] = text in {'1', 'true'}
    return out.reset_index()

if DEMO_MODE:
    DATA, STATIONS = synthetic_inputs()
else:
    if METADATA_PATH is None:
        candidates = [OUTPUT_ROOT / 'DataExtract.csv', PROJECT_DIR / 'DataExtract.csv', Path('DataExtract.csv')]
        METADATA_PATH = next((p for p in candidates if p.is_file()), None)
    if METADATA_PATH is None:
        raise FileNotFoundError('Provide DataExtract.csv and set METADATA_PATH in Settings.')
    STATIONS = load_metadata(METADATA_PATH)
    EEA_FOLDER = download_eea_inputs()
    observed = load_eea_observations(EEA_FOLDER, STATIONS)
    observed, STATIONS = restrict_station_coverage(observed, STATIONS)
    raw_paths = download_cams_inputs()
    cams_daily = cams_daily_at_stations(raw_paths, STATIONS)
    # Full calendar also tracks references with missing observations on a dust day.
    calendar_index = pd.MultiIndex.from_product([
        STATIONS['Samplingpoint'], pd.date_range(CONTEXT_START, CONTEXT_END)
    ], names=['Samplingpoint', 'day'])
    DATA = calendar_index.to_frame(index=False).merge(observed, on=['Samplingpoint', 'day'], how='left', validate='one_to_one')
    DATA = DATA.merge(cams_daily, on=['Samplingpoint', 'day'], how='left', validate='one_to_one')
    del observed, cams_daily

DATA, STATIONS = restrict_station_coverage(DATA, STATIONS)
STATIONS = apply_station_rules(STATIONS, STATION_RULES_CSV)

def apply_dust_screen(data, overrides_path=None):
    out = data.copy()
    screen = pd.Series(pd.NA, index=out.index, dtype='boolean')
    available = out['cams_dust'].notna() & np.isfinite(out['cams_dust'])
    screen.loc[available] = out.loc[available, 'cams_dust'].gt(CAMS_DUST_THRESHOLD).to_numpy()
    out['candidate_dust_day'] = screen
    out['screen_source'] = np.where(available, 'CAMS threshold (provisional)', 'unknown')
    out['evidence_note'] = ''
    if overrides_path is not None:
        decisions = pd.read_csv(overrides_path, dtype={'Samplingpoint': str})
        required = {'Samplingpoint', 'day', 'is_dust_candidate', 'evidence_note'}
        if not required.issubset(decisions):
            raise ValueError(f'Evidence CSV requires {required}')
        decisions['day'] = local_calendar_dates(decisions['day'])
        if decisions.duplicated(['Samplingpoint', 'day']).any():
            raise ValueError('Duplicate daily evidence decisions.')
        out = out.merge(decisions.rename(columns={'evidence_note': '_note'}), on=['Samplingpoint', 'day'], how='left', validate='one_to_one')
        supplied = out['is_dust_candidate'].notna()
        vals = out.loc[supplied, 'is_dust_candidate'].astype(str).str.lower()
        mapping = {'1': True, '0': False, '1.0': True, '0.0': False, 'true': True, 'false': False}
        if not vals.isin(mapping).all():
            raise ValueError('Evidence is_dust_candidate must be 0 or 1.')
        if out.loc[supplied, '_note'].fillna('').str.strip().eq('').any():
            raise ValueError('Every manual evidence decision needs an evidence_note.')
        out.loc[supplied, 'candidate_dust_day'] = vals.map(mapping).to_numpy()
        out.loc[supplied, 'screen_source'] = 'documented override'
        out.loc[supplied, 'evidence_note'] = out.loc[supplied, '_note']
        out = out.drop(columns=['is_dust_candidate', '_note'])
    return out

DATA = apply_dust_screen(DATA, EPISODE_OVERRIDES_CSV)
print(f'{len(STATIONS):,} stations · {len(DATA):,} station-days including context')
display(STATIONS[['Samplingpoint', 'Area', 'Type', 'coverage_pct', 'Timezone']].head(12))

## 6 · Automatic assignment and validation
**Priority:** nearest eligible Option 1 reference → own Option 2 background
series → nearest eligible urban/suburban Option 2 reference → unassigned.

Distances are great-circle distances. Automatic pairing uses the configured
radii and optional altitude/country filters. Every pair must have the same
daily UTC window. The map can override the distance/altitude/country filters
with a written reason, but cannot turn a traffic site into a background
reference or transfer mismatched daily windows.

Missing reference data on a particular date stays unavailable for that date:
Option 2 fallback is an assignment decision, not a hidden daily data fill.

In [ ]:
EARTH_KM = 6371.0088

def xyz(stations):
    lat = np.radians(stations['Latitude'].to_numpy(float))
    lon = np.radians(stations['Longitude'].to_numpy(float))
    return np.column_stack([np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)])

def haversine_km(lat1, lon1, lat2, lon2):
    a, b, c, d = np.radians([lat1, lon1, lat2, lon2])
    h = np.sin((c-a)/2)**2 + np.cos(a)*np.cos(c)*np.sin((d-b)/2)**2
    return float(2 * EARTH_KM * np.arcsin(np.sqrt(np.clip(h, 0, 1))))

def eligible_references(stations, option):
    background = stations['Type'].eq('background') & ~stations['is_hotspot'].astype(bool)
    coordinates = (stations['Latitude'].between(-90, 90) & stations['Longitude'].between(-180, 360)
                   & stations['tz_offset_min'].notna())
    if option == 'option1':
        return background & coordinates & stations['Area'].eq('rural') & stations['regional_enabled'].astype(bool)
    return background & coordinates & stations['Area'].isin(['rural', 'suburban', 'urban']) & stations['option2_enabled'].astype(bool)

def pair_metrics(stations, target, ref, method):
    s = stations.set_index('Samplingpoint')
    t, r = s.loc[target], s.loc[ref]
    distance = haversine_km(t.Latitude, t.Longitude, r.Latitude, r.Longitude)
    delta = abs(t.Altitude - r.Altitude) if pd.notna(t.Altitude) and pd.notna(r.Altitude) else np.nan
    flags = []
    radius = OPTION1_RADIUS_KM if method == 'option1' else OPTION2_RADIUS_KM
    if distance > radius:
        flags.append('outside automatic radius')
    if MAX_ALTITUDE_DIFFERENCE_M is not None and (pd.isna(delta) or delta > MAX_ALTITUDE_DIFFERENCE_M):
        flags.append('altitude criterion not met')
    if SAME_COUNTRY_ONLY and t.Country != r.Country:
        flags.append('different country')
    return distance, delta, '; '.join(flags)

def validate_pair(stations, target, reference, method):
    s = stations.set_index('Samplingpoint')
    if target not in s.index or reference not in s.index:
        raise ValueError('Target/reference station is not present in the current input.')
    t, r = s.loc[target], s.loc[reference]
    if pd.isna(t.tz_offset_min) or pd.isna(r.tz_offset_min) or t.tz_offset_min != r.tz_offset_min:
        raise ValueError('Target and reference must have the same fixed-offset daily window.')
    if not np.isfinite([t.Latitude, t.Longitude, r.Latitude, r.Longitude]).all():
        raise ValueError('Target and reference require valid coordinates.')
    mask = eligible_references(stations, method)
    if not bool(mask.loc[stations['Samplingpoint'].eq(reference)].iloc[0]):
        raise ValueError('Reference does not meet the enabled background/area criteria.')
    if method == 'option2' and target != reference and r.Area not in {'urban', 'suburban'}:
        raise ValueError('External Option 2 references must be urban/suburban background sites.')
    return pair_metrics(stations, target, reference, method)

def candidate_references(stations, target, method, include_outside=False):
    s = stations.set_index('Samplingpoint')
    t = s.loc[target]
    pool = stations.loc[eligible_references(stations, method)].copy()
    if method == 'option2':
        pool = pool.loc[pool['Samplingpoint'].eq(target) | pool['Area'].isin(['urban', 'suburban'])]
    pool = pool.loc[pool['tz_offset_min'].eq(t.tz_offset_min)].copy()
    columns = ['reference_id', 'distance_km', 'altitude_difference_m', 'criteria_flags']
    if pool.empty or not np.isfinite([t.Latitude, t.Longitude]).all():
        return pd.DataFrame(columns=columns)
    chord = np.linalg.norm(xyz(pool) - xyz(pd.DataFrame([t]))[0], axis=1)
    pool['distance_km'] = 2 * EARTH_KM * np.arcsin(np.clip(chord / 2, 0, 1))
    pool['altitude_difference_m'] = (pool['Altitude'] - t.Altitude).abs()
    radius = OPTION1_RADIUS_KM if method == 'option1' else OPTION2_RADIUS_KM
    flags = []
    for r in pool.itertuples():
        issues = []
        if r.distance_km > radius:
            issues.append('outside automatic radius')
        if MAX_ALTITUDE_DIFFERENCE_M is not None and (pd.isna(r.altitude_difference_m) or r.altitude_difference_m > MAX_ALTITUDE_DIFFERENCE_M):
            issues.append('altitude criterion not met')
        if SAME_COUNTRY_ONLY and r.Country != t.Country:
            issues.append('different country')
        flags.append('; '.join(issues))
    pool['criteria_flags'] = flags
    if not include_outside:
        pool = pool.loc[pool['criteria_flags'].eq('')]
    return pool.rename(columns={'Samplingpoint': 'reference_id'})[columns].sort_values(['distance_km', 'reference_id'])

def nearest_automatic(stations, target_row, pool, tree, radius):
    if tree is None or pd.isna(target_row.tz_offset_min) or not np.isfinite([target_row.Latitude, target_row.Longitude]).all():
        return None
    point = xyz(pd.DataFrame([target_row]))[0]
    chord = 2 * math.sin(min(radius / EARTH_KM, math.pi) / 2)
    candidates = pool.iloc[tree.query_ball_point(point, chord)].copy()
    candidates = candidates.loc[candidates['tz_offset_min'].eq(target_row.tz_offset_min)]
    if SAME_COUNTRY_ONLY:
        candidates = candidates.loc[candidates['Country'].eq(target_row.Country)]
    if MAX_ALTITUDE_DIFFERENCE_M is not None:
        candidates = candidates.loc[(candidates['Altitude'] - target_row.Altitude).abs().le(MAX_ALTITUDE_DIFFERENCE_M)]
    if candidates.empty:
        return None
    distances = np.linalg.norm(xyz(candidates) - point, axis=1)
    candidates['_distance'] = distances
    return candidates.sort_values(['_distance', 'Samplingpoint']).iloc[0]['Samplingpoint']

def automatic_assignments(stations):
    pools = {
        'option1': stations.loc[eligible_references(stations, 'option1')].copy(),
        'option2': stations.loc[eligible_references(stations, 'option2') & stations['Area'].isin(['urban', 'suburban'])].copy(),
    }
    trees = {k: cKDTree(xyz(v)) if len(v) else None for k, v in pools.items()}
    own_option2 = set(stations.loc[eligible_references(stations, 'option2'), 'Samplingpoint'])
    records = []
    for _, target in stations.iterrows():
        ref = nearest_automatic(stations, target, pools['option1'], trees['option1'], OPTION1_RADIUS_KM)
        method = 'option1' if ref is not None else 'unassigned'
        reason = 'nearest regional-background candidate' if ref is not None else 'no eligible regional reference within automatic criteria'
        if ref is None:
            if target.Samplingpoint in own_option2:
                ref, method = target.Samplingpoint, 'option2'
                reason += '; own background series'
            else:
                ref = nearest_automatic(stations, target, pools['option2'], trees['option2'], OPTION2_RADIUS_KM)
                if ref is not None:
                    method = 'option2'
                    reason += '; nearest urban/suburban background'
        distance = altitude = np.nan
        flags = ''
        if ref is not None:
            distance, altitude, flags = validate_pair(stations, target.Samplingpoint, ref, method)
        records.append({'Samplingpoint': target.Samplingpoint, 'method': method,
                        'reference_id': ref or '', 'distance_km': distance, 'altitude_difference_m': altitude,
                        'criteria_flags': flags, 'origin': 'automatic', 'reviewed': False,
                        'reason': reason, 'updated_utc': datetime.now(timezone.utc).isoformat()})
    return pd.DataFrame(records)

LINKS = automatic_assignments(STATIONS)
ASSIGNMENT_AUDIT = []
display(LINKS[['Samplingpoint', 'method', 'reference_id', 'distance_km', 'reviewed']])

## 7 · Estimate reference dust and apply the links
Medians use only finite PM observations whose screening is explicitly clean.
Unknown CAMS days cannot enter the background. The calendar window never grows
to find a fixed number of clean observations. Counts and before/after support
are retained for audit. A zero or negative increment is exported as zero dust,
with the original signed increment retained separately.

For a recipient, the stored reference dust estimate is preserved even if it
exceeds the recipient's measured PM. The optional concentration cap affects
only the applied deduction and is flagged. It is a tool convention, not a rule
asserted by the supplied excerpt. Recipient and reference screening must agree
by default; conflicts remain unavailable for review.

In [ ]:
def reference_dust_series(data, reference_id, method):
    frame = data.loc[data['Samplingpoint'].eq(reference_id)].sort_values('day').copy().reset_index(drop=True)
    half = 15 if method == 'option1' else 3
    min_days = MIN_CLEAN_DAYS_OPTION1 if method == 'option1' else MIN_CLEAN_DAYS_OPTION2
    days = frame['day'].to_numpy(dtype='datetime64[ns]')
    pm = frame['daily_mean'].to_numpy(float)
    flags = frame['candidate_dust_day'].astype('boolean')
    is_dust = flags.fillna(False).to_numpy(bool)
    clean = (flags.eq(False).fillna(False).to_numpy(bool) & np.isfinite(pm))
    medians = np.full(len(frame), np.nan)
    counts = np.zeros(len(frame), dtype='int16')
    before = counts.copy()
    after = counts.copy()
    window_start = np.full(len(frame), np.datetime64('NaT'), dtype='datetime64[ns]')
    window_end = window_start.copy()
    # Contiguous screened event runs, used only by the explicit alternate mode.
    starts = np.arange(len(frame))
    ends = starts.copy()
    for i in range(1, len(frame)):
        if is_dust[i] and is_dust[i-1] and days[i] - days[i-1] == np.timedelta64(1, 'D'):
            starts[i] = starts[i-1]
    for i in range(len(frame)-2, -1, -1):
        if is_dust[i] and is_dust[i+1] and days[i+1] - days[i] == np.timedelta64(1, 'D'):
            ends[i] = ends[i+1]
    for i in np.flatnonzero(is_dust):
        a, b = days[i] - np.timedelta64(half, 'D'), days[i] + np.timedelta64(half, 'D')
        if method == 'option2' and OPTION2_WINDOW_MODE == 'episode_edges':
            a = days[starts[i]] - np.timedelta64(3, 'D')
            b = days[ends[i]] + np.timedelta64(3, 'D')
        left, right = np.searchsorted(days, [a, b], side='left')
        right = np.searchsorted(days, b, side='right')
        ix = np.arange(left, right)
        ix = ix[clean[ix] & (days[ix] != days[i])]
        # All internal event days are screened dust, so cannot enter this window.
        counts[i] = len(ix)
        before[i] = np.sum(days[ix] < days[i])
        after[i] = np.sum(days[ix] > days[i])
        window_start[i], window_end[i] = a, b
        if (counts[i] >= min_days and before[i] >= MIN_CLEAN_DAYS_EACH_SIDE
                and after[i] >= MIN_CLEAN_DAYS_EACH_SIDE):
            medians[i] = np.median(pm[ix])
    unknown = flags.isna().to_numpy()
    status = np.full(len(frame), 'clean_reference', dtype=object)
    status[unknown] = 'unknown_reference_screen'
    status[is_dust & ~np.isfinite(pm)] = 'missing_reference_observation'
    status[is_dust & np.isfinite(pm) & ~np.isfinite(medians)] = 'insufficient_clean_days'
    estimated = is_dust & np.isfinite(pm) & np.isfinite(medians)
    raw = pm - medians
    status[estimated & (raw > 0)] = 'positive_reference_increment'
    status[estimated & (raw <= 0)] = 'no_positive_reference_increment'
    contribution = np.full(len(frame), np.nan)
    contribution[~is_dust & ~unknown & np.isfinite(pm)] = 0.0
    contribution[estimated] = np.maximum(0, raw[estimated])
    return pd.DataFrame({
        'reference_id': reference_id, 'method': method, 'day': frame['day'],
        'reference_pm': pm, 'reference_cams_dust': frame['cams_dust'],
        'reference_candidate': flags, 'reference_baseline': medians,
        'reference_dust_raw': raw, 'reference_dust': contribution,
        'n_clean_days': counts, 'n_clean_before': before, 'n_clean_after': after,
        'window_start': window_start, 'window_end': window_end,
        'half_window_days': half,
        'window_mode': 'per_day' if method == 'option1' else OPTION2_WINDOW_MODE,
        'reference_status': status,
    })

REFERENCE_CACHE = {}

def evaluate_assignments(data, stations, links):
    # Validate every active pair; importing a CSV cannot bypass scientific criteria.
    if links['Samplingpoint'].duplicated().any():
        raise ValueError('Duplicate target assignments.')
    for row in links.itertuples():
        if row.method != 'unassigned':
            validate_pair(stations, row.Samplingpoint, row.reference_id, row.method)
    pairs = links.loc[links['method'].ne('unassigned'), ['reference_id', 'method']].drop_duplicates()
    references = []
    for ref, method in pairs.itertuples(index=False, name=None):
        key = (ref, method)
        if key not in REFERENCE_CACHE:
            REFERENCE_CACHE[key] = reference_dust_series(data, ref, method)
        references.append(REFERENCE_CACHE[key])
    out = data.loc[data['day'].dt.year.eq(YEAR) & data['Samplingpoint'].isin(links['Samplingpoint'])].merge(links, on='Samplingpoint', how='left', validate='many_to_one')
    if references:
        reference_table = pd.concat(references, ignore_index=True)
    else:
        reference_table = reference_dust_series(data, stations.iloc[0]['Samplingpoint'], 'option1').iloc[:0]
    out = out.merge(reference_table, on=['reference_id', 'method', 'day'], how='left', validate='many_to_one')
    target_true = out['candidate_dust_day'].fillna(False).to_numpy(bool)
    target_clean = out['candidate_dust_day'].eq(False).fillna(False).to_numpy(bool)
    ref_true = out['reference_candidate'].eq(True).fillna(False).to_numpy(bool)
    target_pm = out['daily_mean'].to_numpy(float)
    contribution = np.full(len(out), np.nan)
    status = np.full(len(out), 'unknown_target_screen', dtype=object)
    status[target_clean] = 'screened_clean'
    contribution[target_clean] = 0
    possible = target_true if REQUIRE_TARGET_DUST_SCREEN else ~target_clean
    assigned = out['method'].isin(['option1', 'option2']).to_numpy()
    status[possible & ~assigned] = 'no_reference'
    status[possible & assigned] = out.loc[possible & assigned, 'reference_status'].fillna('reference_unavailable').to_numpy()
    mismatch = possible & assigned & ~ref_true
    status[mismatch] = 'target_reference_screen_mismatch'
    rawdust = out['reference_dust'].to_numpy(float)
    usable = possible & assigned & ref_true & np.isfinite(rawdust)
    contribution[usable] = rawdust[usable]
    status[usable] = np.where(rawdust[usable] > 0, 'dust_quantified', 'no_positive_increment')
    cap = np.isfinite(contribution) & np.isfinite(target_pm) & (contribution > target_pm)
    applied = contribution.copy()
    if CAP_DEDUCTION_TO_OBSERVED_PM:
        applied[cap] = target_pm[cap]
    else:
        cap[:] = False
    missing_pm = ~np.isfinite(target_pm)
    status[missing_pm] = 'missing_target_observation'
    applied[missing_pm] = np.nan
    out['Dust_contribution_transferred'] = contribution.astype('float32')
    out['Dust_contribution'] = applied.astype('float32')
    out['deduction_capped'] = cap
    out['corrected_pollutant'] = (target_pm - applied).astype('float32')
    out['calculation_status'] = status
    out['dust_flag'] = out['candidate_dust_day']
    out['dust_confirmed'] = usable & (rawdust > 0)
    out['Exceedance'] = out['daily_mean'].gt(PM_DAILY_LIMIT)
    out['Dust_exceedance'] = out['Exceedance'] & out['corrected_pollutant'].le(PM_DAILY_LIMIT)
    out['eea_resolution'] = EEA_TEMPORAL_FLAG
    out['day_definition'] = 'station local standard day (fixed metadata offset)'
    out['pollutant'] = POLLUTANT
    geo_cols = ['Samplingpoint', 'Latitude', 'Longitude', 'Altitude', 'Area', 'Type', 'Timezone', 'tz_offset_min']
    out = out.merge(stations[geo_cols], on='Samplingpoint', how='left', validate='many_to_one')
    return out

def station_summary(result, links):
    rows = []
    for sid, group in result.groupby('Samplingpoint', sort=True):
        paired = group['daily_mean'].notna() & group['corrected_pollutant'].notna()
        valid_observed = group['daily_mean'].notna()
        n_expected = 366 if calendar.isleap(YEAR) else 365
        rows.append({
            'Samplingpoint': sid, 'observed_days': int(valid_observed.sum()),
            'deduction_available_days': int(paired.sum()), 'calendar_days': n_expected,
            'deduction_coverage_pct': paired.sum() / n_expected * 100,
            'Exceedance_days': int(group.loc[valid_observed, 'Exceedance'].sum()),
            'Dust_exceedance_days': int(group['Dust_exceedance'].sum()),
            'Exceedance_days_paired': int(group.loc[paired, 'Exceedance'].sum()),
            'Corrected_exceedance_days_paired': int(group.loc[paired, 'corrected_pollutant'].gt(PM_DAILY_LIMIT).sum()),
            'Exceedance_days_unassessed': int(group.loc[~paired & valid_observed, 'Exceedance'].sum()),
            'Average_pollutant_all_observed': group.loc[valid_observed, 'daily_mean'].mean(),
            'Average_pollutant_paired': group.loc[paired, 'daily_mean'].mean(),
            'Average_pollutant_dust_removed_paired': group.loc[paired, 'corrected_pollutant'].mean(),
            'Average_dust_deduction_available': group.loc[paired, 'Dust_contribution'].mean(),
            'full_year_dust_mean_available': bool(paired.sum() == n_expected),
            'Capped_days': int(group['deduction_capped'].sum()),
        })
    return pd.DataFrame(rows).merge(links, on='Samplingpoint', validate='one_to_one')

RESULT = evaluate_assignments(DATA, STATIONS, LINKS)
SUMMARY = station_summary(RESULT, LINKS)
display(RESULT['calculation_status'].value_counts().rename_axis('status').to_frame('station-days'))
display(SUMMARY[['Samplingpoint', 'method', 'reference_id', 'deduction_coverage_pct', 'Dust_exceedance_days']])

## 8 · Save, restore and audit
Exports include daily results (Parquet), station summary (CSV), editable links
and eligibility (CSV), change history (JSON), map links (GeoJSON), and run
settings (JSON). Comparisons of observed and corrected annual means use the
**same available days**; incomplete-year means are explicitly labelled.

Saved links are validated against the current station IDs/coordinates/timezones.
References are never automatically changed because one day's observation is
missing. To alter dust screening, edit the evidence CSV, rerun from section 5,
The cache is cleared automatically whenever section 7 is rerun.

In [ ]:
def run_settings():
    return {
        'algorithm_version': ALGORITHM_VERSION, 'demo_mode': DEMO_MODE,
        'year': YEAR, 'pollutant': POLLUTANT, 'dataset': DATASET,
        'eea_temporal_flag': EEA_TEMPORAL_FLAG, 'countries': COUNTRIES,
        'day_definition': 'station local standard day',
        'context_start': CONTEXT_START, 'context_end': CONTEXT_END,
        'option1_radius_km': OPTION1_RADIUS_KM, 'option2_radius_km': OPTION2_RADIUS_KM,
        'max_altitude_difference_m': MAX_ALTITUDE_DIFFERENCE_M,
        'same_country_only': SAME_COUNTRY_ONLY,
        'option1_half_window_days': 15, 'option2_half_window_days': 3,
        'option2_window_mode': OPTION2_WINDOW_MODE,
        'min_clean_days_option1': MIN_CLEAN_DAYS_OPTION1,
        'min_clean_days_option2': MIN_CLEAN_DAYS_OPTION2,
        'min_clean_days_each_side': MIN_CLEAN_DAYS_EACH_SIDE,
        'cams_dust_threshold_ug_m3': CAMS_DUST_THRESHOLD,
        'min_obs_hours': MIN_OBS_HOURS, 'min_cams_hours': MIN_CAMS_HOURS,
        'min_station_coverage_pct': MIN_STATION_COVERAGE_PCT,
        'pm_daily_limit_ug_m3': PM_DAILY_LIMIT,
        'require_target_dust_screen': REQUIRE_TARGET_DUST_SCREEN,
        'cap_deduction_to_observed': CAP_DEDUCTION_TO_OBSERVED_PM,
        'guideline_basis': 'User-supplied excerpt §§5.2.2–5.2.3, Boxes 7–11; version/date not supplied',
        'screening_note': 'CAMS threshold alone is a provisional candidate screen; validate model threshold and document natural-origin evidence.',
    }

def pairing_context(stations):
    columns = ['Samplingpoint', 'Latitude', 'Longitude', 'tz_offset_min', 'Area', 'Type']
    return {'year': YEAR, 'pollutant': POLLUTANT, 'dataset': DATASET,
            'eea_temporal_flag': EEA_TEMPORAL_FLAG,
            'stations_hash': digest(stations[columns].sort_values('Samplingpoint').to_dict('records'))}

def write_links_only():
    LINKS.to_csv(RUN_DIR / 'station_links.csv', index=False)
    STATIONS[['Samplingpoint', 'regional_enabled', 'option2_enabled', 'is_hotspot']].to_csv(RUN_DIR / 'station_rules.csv', index=False)
    atomic_json(RUN_DIR / 'station_links.context.json', pairing_context(STATIONS))
    atomic_json(RUN_DIR / 'assignment_audit.json', ASSIGNMENT_AUDIT)

def restore_links():
    global STATIONS, LINKS, ASSIGNMENT_AUDIT
    path = RUN_DIR / 'station_links.csv'
    context = RUN_DIR / 'station_links.context.json'
    if not path.exists() or not context.exists():
        return False
    if json.loads(context.read_text()) != pairing_context(STATIONS):
        raise ValueError('Saved links belong to different station metadata or settings. Use a separate RUN_DIR or review the CSV manually.')
    restored_stations = apply_station_rules(STATIONS, RUN_DIR / 'station_rules.csv') if (RUN_DIR / 'station_rules.csv').exists() else STATIONS.copy()
    loaded = pd.read_csv(path, dtype={'Samplingpoint': str, 'reference_id': str}, keep_default_na=False)
    if set(loaded['Samplingpoint']) != set(restored_stations['Samplingpoint']) or loaded['Samplingpoint'].duplicated().any():
        raise ValueError('Saved assignments do not match the current target station set.')
    for i, row in loaded.iterrows():
        if row['method'] not in {'option1', 'option2', 'unassigned'}:
            raise ValueError('Invalid method in saved links.')
        if row['method'] != 'unassigned':
            distance, altitude, flags = validate_pair(restored_stations, row['Samplingpoint'], row['reference_id'], row['method'])
            if flags and not str(row.get('reason', '')).strip():
                raise ValueError('Saved manual override lacks a reason.')
            loaded.loc[i, 'distance_km'] = distance
            loaded.loc[i, 'altitude_difference_m'] = altitude
            loaded.loc[i, 'criteria_flags'] = flags
    loaded['reviewed'] = loaded['reviewed'].astype(str).str.lower().isin(['true', '1'])
    STATIONS, LINKS = restored_stations, loaded
    audit_path = RUN_DIR / 'assignment_audit.json'
    ASSIGNMENT_AUDIT = json.loads(audit_path.read_text()) if audit_path.exists() else []
    return True

def links_geojson():
    lookup = STATIONS.set_index('Samplingpoint')
    features = []
    for row in LINKS.itertuples():
        target = lookup.loc[row.Samplingpoint]
        if not np.isfinite([target.Longitude, target.Latitude]).all():
            continue
        props = {'Samplingpoint': row.Samplingpoint, 'method': row.method,
                 'reference_id': row.reference_id, 'origin': row.origin, 'reviewed': bool(row.reviewed)}
        features.append({'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [float(target.Longitude), float(target.Latitude)]}, 'properties': props})
        if row.method != 'unassigned' and row.reference_id != row.Samplingpoint:
            ref = lookup.loc[row.reference_id]
            features.append({'type': 'Feature', 'geometry': {'type': 'LineString', 'coordinates': [
                [float(target.Longitude), float(target.Latitude)], [float(ref.Longitude), float(ref.Latitude)]
            ]}, 'properties': props})
    return {'type': 'FeatureCollection', 'features': features}

def export_results():
    global SUMMARY
    SUMMARY = station_summary(RESULT, LINKS)
    RESULT.to_parquet(RUN_DIR / 'daily_dust_results.parquet', index=False)
    SUMMARY.to_csv(RUN_DIR / 'station_summary.csv', index=False)
    write_links_only()
    atomic_json(RUN_DIR / 'station_links.geojson', links_geojson())
    atomic_json(RUN_DIR / 'run_settings.json', {**run_settings(), 'exported_utc': datetime.now(timezone.utc).isoformat()})
    if not (RUN_DIR / 'episode_evidence_template.csv').exists():
        pd.DataFrame(columns=['Samplingpoint', 'day', 'is_dust_candidate', 'evidence_note']).to_csv(RUN_DIR / 'episode_evidence_template.csv', index=False)
    return RUN_DIR

if AUTO_LOAD_SAVED_LINKS and restore_links():
    REFERENCE_CACHE.clear()
    RESULT = evaluate_assignments(DATA, STATIONS, LINKS)
    SUMMARY = station_summary(RESULT, LINKS)
    print('Restored and validated saved assignments and eligibility decisions.')

## 9 · Interactive reference map
**Inspect:** click a station. Its current reference and represented stations
are highlighted. Use the search box for large networks.

**Change a link:** select the recipient → choose Option 1/2 → choose a reference
in the list, or switch to **Pick reference** and click a candidate → **Apply**.
A different eligible reference beyond the automatic radius can be selected by
enabling **Show outside criteria**, with a reason. Different daily timezones
cannot be linked. The selected reference's distance/altitude are shown.

**Reference eligibility:** select a reference, change its eligibility flags,
then apply them. Existing links are rebuilt automatically where necessary;
manual links that remain valid are retained. Eligibility edits are audited.

**Apply** recalculates the selected recipient and autosaves links. Use
**Export all results** after editing to refresh the full result files.
**Reset selected** restores that station's automatic assignment.

In [ ]:
import ipywidgets as W
from ipyleaflet import Map, GeoJSON, LayerGroup, Polyline, Circle, basemaps

METHOD_COLORS = {'option1': '#2563eb', 'option2': '#e68a12', 'unassigned': '#94a3b8'}

class ReferenceEditor:
    def __init__(self):
        self._updating = False
        self.search = W.Text(placeholder='Search station ID', description='Find:')
        options = [(s, s) for s in sorted(STATIONS['Samplingpoint'])[:500]]
        self.target = W.Dropdown(options=options, description='Station:', layout=W.Layout(width='100%'))
        self.method = W.Dropdown(options=[('Option 1 · regional', 'option1'), ('Option 2 · background', 'option2'), ('Unassigned', 'unassigned')], description='Method:')
        self.reference = W.Dropdown(options=[], description='Reference:', layout=W.Layout(width='100%'))
        self.outside = W.Checkbox(value=False, description='Show outside automatic criteria', indent=False)
        self.mode = W.ToggleButtons(options=[('Inspect', 'inspect'), ('Pick reference', 'pick')], value='inspect')
        self.note = W.Textarea(description='Reason:', placeholder='Representativeness / reason for changing the automatic proposal', layout=W.Layout(width='100%', height='65px'))
        self.reviewed = W.Checkbox(value=True, description='I have reviewed this assignment', indent=False)
        self.apply = W.Button(description='Apply reference', button_style='primary', icon='check')
        self.reset = W.Button(description='Reset selected', icon='undo')
        self.save = W.Button(description='Export all results', button_style='success', icon='save')
        self.load = W.Button(description='Reload saved links', icon='folder-open')
        self.info = W.HTML()
        self.candidate_info = W.HTML()
        self.status = W.HTML()
        self.stats = W.HTML()
        self.regional = W.Checkbox(description='Eligible as Option 1 reference', indent=False)
        self.local = W.Checkbox(description='Eligible as Option 2 reference', indent=False)
        self.hotspot = W.Checkbox(description='Hotspot / exclude as reference', indent=False)
        self.rules_button = W.Button(description='Apply eligibility', icon='sliders')
        self.plot_out = W.Output(layout=W.Layout(max_height='460px', overflow='auto'))
        valid = STATIONS.dropna(subset=['Latitude', 'Longitude'])
        center = (float(valid.Latitude.median()), float(valid.Longitude.median())) if len(valid) else (50, 10)
        self.map = Map(center=center, zoom=7 if DEMO_MODE else 5, basemap=basemaps.OpenStreetMap.Mapnik,
                       scroll_wheel_zoom=True, prefer_canvas=True, layout=W.Layout(width='100%', height='560px'))
        self.points = GeoJSON(data={'type': 'FeatureCollection', 'features': []},
                              point_style={'radius': 6, 'weight': 1.5, 'fillOpacity': .85},
                              style_callback=lambda f: f['properties']['style'])
        self.connections = GeoJSON(data={'type': 'FeatureCollection', 'features': []},
                                   style={'color': '#64748b', 'weight': 2, 'opacity': .65})
        self.selection = LayerGroup()
        self.map.add(self.connections)
        self.map.add(self.points)
        self.map.add(self.selection)
        self.points.on_click(self._map_click)
        self.search.observe(self._search, names='value')
        self.target.observe(self._target_changed, names='value')
        self.method.observe(self._candidates_changed, names='value')
        self.outside.observe(self._candidates_changed, names='value')
        self.reference.observe(self._reference_changed, names='value')
        self.apply.on_click(self._apply)
        self.reset.on_click(self._reset)
        self.save.on_click(self._save)
        self.load.on_click(self._load)
        self.rules_button.on_click(self._rules)
        controls = W.VBox([
            self.search, self.target, self.mode, self.info,
            self.method, self.outside, self.reference, self.candidate_info,
            self.note, self.reviewed, W.HBox([self.apply, self.reset]),
            W.HTML('<hr><b>Selected station: reference eligibility</b>'),
            self.regional, self.local, self.hotspot, self.rules_button,
        ], layout=W.Layout(width='370px', min_width='310px', padding='10px'))
        legend = W.HTML('<b style="color:#2563eb">● Option 1</b> &nbsp; <b style="color:#e68a12">● Option 2</b> &nbsp; <span style="color:#94a3b8">● Unassigned</span> &nbsp; Thick outlines: eligible references. Pink: selected station.')
        self.widget = W.VBox([self.stats, legend, W.HBox([controls, self.map], layout=W.Layout(width='100%')),
                              W.HBox([self.save, self.load]), self.status, self.plot_out])
        self.refresh_points()
        self.select(self.target.value)

    def message(self, value, error=False):
        color = '#b91c1c' if error else '#166534'
        self.status.value = f'<div style="padding:8px;color:{color}">{html.escape(str(value))}</div>'

    def _search(self, change):
        query = change['new'].strip().lower()
        ids = sorted(s for s in STATIONS['Samplingpoint'] if query in s.lower())[:500]
        if self.target.value and self.target.value not in ids:
            ids.insert(0, self.target.value)
        old = self.target.value
        self._updating = True
        self.target.options = [(s, s) for s in ids]
        if old in ids:
            self.target.value = old
        self._updating = False

    def select(self, sid):
        if sid is None:
            return
        self._updating = True
        if sid not in [value for label, value in self.target.options]:
            self.target.options = list(self.target.options) + [(sid, sid)]
        self.target.value = sid
        row = LINKS.set_index('Samplingpoint').loc[sid]
        station = STATIONS.set_index('Samplingpoint').loc[sid]
        self.method.value = row.method
        self.note.value = ''
        self.reviewed.value = True
        self.regional.value = bool(station.regional_enabled)
        self.local.value = bool(station.option2_enabled)
        self.hotspot.value = bool(station.is_hotspot)
        self._updating = False
        self.populate_candidates(row.reference_id)
        n = int(LINKS['reference_id'].eq(sid).sum())
        self.info.value = (
            f'<b>{html.escape(sid)}</b><br>{html.escape(station.Area)} / {html.escape(station.Type)} · '
            f'{html.escape(str(station.Timezone))}<br>Current: <b>{html.escape(row.method)}</b> → '
            f'{html.escape(row.reference_id or "no reference")}<br>Origin: {html.escape(row.origin)} · '
            f'Reviewed: {bool(row.reviewed)}<br>Represents <b>{n}</b> station(s).'
        )
        self.highlight(sid)
        self.plot(sid)

    def _target_changed(self, change):
        if not self._updating:
            self.select(change['new'])

    def _candidates_changed(self, change):
        if not self._updating:
            self.populate_candidates()

    def populate_candidates(self, current=None):
        self._updating = True
        if self.method.value == 'unassigned':
            self.reference.options = []
            self.reference.disabled = True
        else:
            candidates = candidate_references(STATIONS, self.target.value, self.method.value, self.outside.value)
            self.reference.disabled = candidates.empty
            options = [(f'{r.reference_id} · {r.distance_km:.1f} km' + (' ⚑' if r.criteria_flags else ''), r.reference_id)
                       for r in candidates.itertuples()]
            self.reference.options = options
            ids = [v for label, v in options]
            if current in ids:
                self.reference.value = current
        self._updating = False
        self._reference_changed(None)

    def _reference_changed(self, change):
        if self._updating:
            return
        ref = self.reference.value
        if not ref or self.method.value == 'unassigned':
            self.candidate_info.value = '<i>No eligible candidate selected.</i>'
            return
        distance, altitude, flags = validate_pair(STATIONS, self.target.value, ref, self.method.value)
        alt = f'{altitude:.0f} m' if np.isfinite(altitude) else 'unknown'
        self.candidate_info.value = f'<b>Candidate:</b> {distance:.1f} km · altitude difference {alt}<br>{html.escape(flags)}'

    def _map_click(self, **event):
        feature = event.get('feature', {})
        sid = feature.get('properties', {}).get('Samplingpoint')
        if not sid:
            return
        if self.mode.value == 'inspect':
            self.select(sid)
        elif self.method.value != 'unassigned':
            candidates = [v for label, v in self.reference.options]
            if sid in candidates:
                self.reference.value = sid
                self.message('Reference selected. Press Apply reference to update this station.')
            else:
                self.message('Not an eligible reference for this method/window. Inspect eligibility or enable outside criteria.', True)

    def refresh_points(self):
        features = []
        regional = set(STATIONS.loc[eligible_references(STATIONS, 'option1'), 'Samplingpoint'])
        local = set(STATIONS.loc[eligible_references(STATIONS, 'option2'), 'Samplingpoint'])
        assignments = LINKS.set_index('Samplingpoint')
        for row in STATIONS.itertuples():
            if not np.isfinite([row.Latitude, row.Longitude]).all():
                continue
            link = assignments.loc[row.Samplingpoint]
            style = {'fillColor': METHOD_COLORS[link.method], 'color': '#0f172a' if row.Samplingpoint in regional else '#854d0e' if row.Samplingpoint in local else '#ffffff',
                     'weight': 3 if row.Samplingpoint in regional or row.Samplingpoint in local else 1,
                     'fillOpacity': .9, 'radius': 7 if row.Samplingpoint in regional else 5}
            features.append({'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [float(row.Longitude), float(row.Latitude)]},
                             'properties': {'Samplingpoint': row.Samplingpoint, 'method': link.method, 'style': style}})
        self.points.data = {'type': 'FeatureCollection', 'features': features}
        counts = LINKS['method'].value_counts()
        self.stats.value = (
            f'<h3>{"Synthetic demo · " if DEMO_MODE else ""}{len(STATIONS):,} stations</h3>'
            f'Option 1: <b>{counts.get("option1",0)}</b> · Option 2: <b>{counts.get("option2",0)}</b> · '
            f'Unassigned: <b>{counts.get("unassigned",0)}</b> · Reviewed: <b>{int(LINKS.reviewed.sum())}</b>'
        )

    def highlight(self, sid):
        lookup = STATIONS.set_index('Samplingpoint')
        linked = LINKS.loc[LINKS['reference_id'].eq(sid)].copy()
        own = LINKS.loc[LINKS['Samplingpoint'].eq(sid)]
        relevant = pd.concat([linked, own]).drop_duplicates('Samplingpoint')
        features = []
        for row in relevant.itertuples():
            if row.method == 'unassigned' or row.Samplingpoint == row.reference_id:
                continue
            a, b = lookup.loc[row.Samplingpoint], lookup.loc[row.reference_id]
            features.append({'type': 'Feature', 'geometry': {'type': 'LineString', 'coordinates': [[float(a.Longitude), float(a.Latitude)], [float(b.Longitude), float(b.Latitude)]]}, 'properties': {}})
        self.connections.data = {'type': 'FeatureCollection', 'features': features}
        station = lookup.loc[sid]
        layers = []
        if np.isfinite([station.Latitude, station.Longitude]).all():
            layers.append(Circle(location=(station.Latitude, station.Longitude), radius=850,
                                 color='#db2777', fill_color='#db2777', fill_opacity=.35, weight=3))
            if bool(eligible_references(STATIONS, 'option1').loc[STATIONS.Samplingpoint.eq(sid)].iloc[0]):
                layers.append(Circle(location=(station.Latitude, station.Longitude), radius=OPTION1_RADIUS_KM*1000,
                                     color='#2563eb', fill_opacity=.025, weight=1, dash_array='6 4'))
        self.selection.layers = tuple(layers)

    def _apply(self, button):
        global LINKS, RESULT, SUMMARY
        try:
            sid, method, ref = self.target.value, self.method.value, self.reference.value
            old = LINKS.loc[LINKS.Samplingpoint.eq(sid)].iloc[0].to_dict()
            distance = altitude = np.nan
            flags = ''
            if method != 'unassigned':
                if not ref:
                    raise ValueError('Select an eligible reference first.')
                distance, altitude, flags = validate_pair(STATIONS, sid, ref, method)
            else:
                ref = ''
            reason = self.note.value.strip()
            if (flags or (method == 'option2' and old['method'] == 'option1')) and not reason:
                raise ValueError('Add a reason for overriding automatic criteria or replacing Option 1 with Option 2.')
            new = dict(old, method=method, reference_id=ref, distance_km=distance,
                       altitude_difference_m=altitude, criteria_flags=flags, origin='manual',
                       reviewed=self.reviewed.value, reason=reason or (old.get('reason') if (old['method'], old['reference_id']) == (method, ref) else 'manual reference selection; see review status') or 'manual reference review',
                       updated_utc=datetime.now(timezone.utc).isoformat())
            single = pd.DataFrame([new])
            recalculated = evaluate_assignments(DATA, STATIONS, single)
            LINKS.loc[LINKS.Samplingpoint.eq(sid), single.columns] = single.to_numpy()
            RESULT = pd.concat([RESULT.loc[RESULT.Samplingpoint.ne(sid)], recalculated], ignore_index=True)
            SUMMARY = station_summary(RESULT, LINKS)
            ASSIGNMENT_AUDIT.append({'action': 'manual_assignment', 'before': old, 'after': new})
            write_links_only()
            self.refresh_points()
            self.select(sid)
            self.message('Assignment applied and saved; selected station recalculated. Export all results to update output tables.')
        except Exception as exc:
            self.message(str(exc), True)

    def _reset(self, button):
        global LINKS, RESULT, SUMMARY
        sid = self.target.value
        old = LINKS.loc[LINKS.Samplingpoint.eq(sid)].iloc[0].to_dict()
        new = automatic_assignments(STATIONS).loc[lambda x: x.Samplingpoint.eq(sid)]
        LINKS.loc[LINKS.Samplingpoint.eq(sid), new.columns] = new.to_numpy()
        recalculated = evaluate_assignments(DATA, STATIONS, new)
        RESULT = pd.concat([RESULT.loc[RESULT.Samplingpoint.ne(sid)], recalculated], ignore_index=True)
        SUMMARY = station_summary(RESULT, LINKS)
        ASSIGNMENT_AUDIT.append({'action': 'reset_selected', 'before': old, 'after': new.iloc[0].to_dict()})
        write_links_only()
        self.refresh_points()
        self.select(sid)
        self.message('Restored the automatic proposal for this station.')

    def _rules(self, button):
        global STATIONS, LINKS, RESULT, SUMMARY
        try:
            sid = self.target.value
            candidate_stations = STATIONS.copy()
            ix = candidate_stations.Samplingpoint.eq(sid)
            old_rules = STATIONS.loc[ix].iloc[0].to_dict()
            candidate_stations.loc[ix, ['regional_enabled', 'option2_enabled', 'is_hotspot']] = [self.regional.value, self.local.value, self.hotspot.value]
            row = candidate_stations.loc[ix].iloc[0]
            if row.regional_enabled and not (row.Area == 'rural' and row.Type == 'background' and not row.is_hotspot):
                raise ValueError('Option 1 eligibility requires rural background and no hotspot flag.')
            if row.option2_enabled and not (row.Type == 'background' and not row.is_hotspot):
                raise ValueError('Option 2 eligibility requires background and no hotspot flag.')
            rebuilt = automatic_assignments(candidate_stations)
            for old in LINKS.loc[LINKS.origin.eq('manual')].to_dict('records'):
                try:
                    if old['method'] != 'unassigned':
                        validate_pair(candidate_stations, old['Samplingpoint'], old['reference_id'], old['method'])
                    rebuilt.loc[rebuilt.Samplingpoint.eq(old['Samplingpoint']), rebuilt.columns] = pd.DataFrame([old])[rebuilt.columns].to_numpy()
                except ValueError:
                    pass  # invalid manual pair is replaced and recorded below
            before_links = LINKS.to_dict('records')
            new_result = evaluate_assignments(DATA, candidate_stations, rebuilt)
            STATIONS, LINKS, RESULT = candidate_stations, rebuilt, new_result
            SUMMARY = station_summary(RESULT, LINKS)
            ASSIGNMENT_AUDIT.append({'action': 'eligibility_edit', 'station': sid, 'before': old_rules,
                                     'after': row.to_dict(), 'links_before': before_links,
                                     'links_after': LINKS.to_dict('records'),
                                     'utc': datetime.now(timezone.utc).isoformat()})
            write_links_only()
            self.refresh_points()
            self.select(sid)
            self.message('Eligibility saved. Automatic links rebuilt; valid manual links retained. Export all results when finished.')
        except Exception as exc:
            self.message(str(exc), True)

    def _save(self, button):
        try:
            export_results()
            self.message(f'All results exported to {RUN_DIR.resolve()}')
        except Exception as exc:
            self.message(str(exc), True)

    def _load(self, button):
        global RESULT, SUMMARY
        try:
            if restore_links():
                RESULT = evaluate_assignments(DATA, STATIONS, LINKS)
                SUMMARY = station_summary(RESULT, LINKS)
                self.refresh_points()
                self.select(self.target.value)
                self.message('Saved assignments restored and results recalculated.')
            else:
                self.message('No saved assignment table in this run folder.', True)
        except Exception as exc:
            self.message(str(exc), True)

    def plot(self, sid):
        import matplotlib.pyplot as plt
        with self.plot_out:
            clear_output(wait=True)
            part = RESULT.loc[RESULT.Samplingpoint.eq(sid)].sort_values('day')
            fig, ax = plt.subplots(2, 1, figsize=(12, 4.2), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
            ax[0].plot(part.day, part.daily_mean, color='#64748b', lw=.9, label='Observed PM')
            ax[0].plot(part.day, part.corrected_pollutant, color='#2563eb', lw=1, label='Corrected PM')
            ax[0].axhline(PM_DAILY_LIMIT, color='#b91c1c', ls=':', lw=.8, label='Selected daily limit')
            ax[0].set_ylabel('PM (µg/m³)')
            ax[0].legend(loc='upper right', ncol=3, fontsize=8)
            ax[1].plot(part.day, part.Dust_contribution, color='#d97706', lw=1, label='Applied dust deduction')
            missing = part.calculation_status.isin(['no_reference', 'insufficient_clean_days', 'unknown_target_screen', 'missing_reference_observation', 'target_reference_screen_mismatch'])
            ax[1].scatter(part.loc[missing, 'day'], np.zeros(missing.sum()), color='#be123c', marker='x', s=10, label='Unavailable / review')
            ax[1].set_ylabel('Dust (µg/m³)')
            ax[1].legend(loc='upper right', fontsize=8)
            fig.suptitle(('SYNTHETIC · ' if DEMO_MODE else '') + sid, fontsize=11)
            fig.tight_layout()
            display(fig)
            plt.close(fig)
            represented = LINKS.loc[LINKS.reference_id.eq(sid), ['Samplingpoint', 'method', 'distance_km', 'reviewed']]
            if not represented.empty:
                print(f'Stations represented by {sid}: {len(represented)}')
                display(represented.head(30))

EDITOR = ReferenceEditor()
display(EDITOR.widget)

## 10 · Export and inspect results
Rerun this cell after map edits, or press **Export all results** above.
To download the whole output folder on Colab, run the optional cell below.
With Drive enabled, results are already in the configured Drive folder.

In [ ]:
export_results()
print('Exported files:')
for file in sorted(RUN_DIR.iterdir()):
    print(' •', file.name)
display(SUMMARY[['Samplingpoint', 'method', 'reference_id', 'distance_km',
                 'deduction_coverage_pct', 'Exceedance_days', 'Dust_exceedance_days',
                 'Exceedance_days_unassessed']])

In [ ]:
# Optional: set True to download current outputs as one ZIP in Google Colab.
DOWNLOAD_RESULTS_ZIP = False
if DOWNLOAD_RESULTS_ZIP:
    import shutil
    export_results()
    archive_path = shutil.make_archive(str(RUN_DIR) + '_results', 'zip', root_dir=RUN_DIR)
    try:
        from google.colab import files
        files.download(archive_path)
    except ImportError:
        print('Results archive:', archive_path)

## Checks and interpretation
- Positive estimated dust confirms the **quantitative** increment, but natural
  origin and spatial representativeness still need the supporting evidence
  described in §5.2.2. The initial 5 µg/m³ CAMS screen is not automatically a
  validated model-specific detection threshold.
- Audit `n_clean_days`, `window_start/end`, `reference_status`, and
  `calculation_status`. Long episodes may leave no robust Option 2 baseline.
- A fallback reference is not a substitute for representativeness assessment.
  The map makes distance-based proposals reviewable; it does not establish
  similarity of terrain, atmospheric mixing or episode exposure by itself.
- The daily limit is an **analysis setting**, not a claim of legal applicability.
  Set the pollutant/year/assessment framework appropriate to your study.
- Incomplete annual summaries must not be interpreted as a complete annual
  dust deduction or a final compliance determination.
- To use several reference sites as an averaged regional series (also allowed
  by the excerpt), a separate reference-group definition is needed. This
  version deliberately implements one explicitly selected reference per target.

### Sources and provenance
- User-supplied excerpt: §§5.2.2–5.2.3, Boxes 7–11; publication/revision date not
  provided. Reconcile the two multi-day Option 2 descriptions with the final
  adopted guidance before using this for official reporting.
- [EEA download service](https://eeadmz1-downloads-webapp.azurewebsites.net/):
  hourly export in UTC+1; daily data in the timezone reported by countries.
- [EEA e-Reporting schema](https://www.eionet.europa.eu/aqportal/doc/xsd/AirQualityReporting.html):
  recommends local standard time without daylight-saving adjustments.
- [CAMS European air quality reanalyses](https://ads.atmosphere.copernicus.eu/datasets/cams-europe-air-quality-reanalyses?tab=overview)
  ([catalogue](https://ads.atmosphere.copernicus.eu/stac-browser/collections/cams-europe-air-quality-reanalyses)):
  hourly UTC, DOI 10.24381/7cc0465a.
- [ipyleaflet](https://ipyleaflet.readthedocs.io/) for the editable map.

This notebook contains all computational and map code; it does not import
`util.py` or `cams_data_to_eea_daily.py` from an external repository.